# Neural Networks (MLP) — Decision Boundaries & Regularization
*Notebook last updated: 2026-02-25*

This notebook reproduces and extends textbook-style experiments with **multilayer perceptrons (MLPs)** in scikit-learn. The focus is on *interpretable experiments*: how architecture, activation, and regularization change the learned decision boundary.

**Key ideas**
- Logistic regression is a neural network with **no hidden layer**.
- Adding hidden layers introduces **nonlinear feature learning**.
- Feature scaling is essential for stable optimization in MLPs.


## 1. Setup
We keep imports and random seeds centralized for reproducibility.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

# Optional: used by some textbooks for convenient boundary plots
import mglearn

np.random.seed(0)


## 2. A helper: decision boundary plotting
Matplotlib works best when plotting functions accept an `ax` argument, so the function can be used inside subplot grids.

In [ ]:
def plot_decision_boundary(model, X, y, *, ax=None, resolution=300, proba=False,
                           title=None, xlabel="feature 1", ylabel="feature 2"):
    """Plot a 2D decision boundary for a fitted classifier.

    Parameters
    ----------
    model : fitted sklearn classifier
    X : array-like of shape (n_samples, 2)
    y : array-like of shape (n_samples,)
    ax : matplotlib axis; if None, one will be created
    resolution : int, grid density
    proba : bool; if True, uses predict_proba for smooth boundary
    title : str; axis title
    """
    import numpy as np
    import matplotlib.pyplot as plt

    if ax is None:
        _, ax = plt.subplots(figsize=(6, 5))

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, resolution),
        np.linspace(y_min, y_max, resolution),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    if proba and hasattr(model, "predict_proba"):
        Z = model.predict_proba(grid)[:, 1]
        Z = Z.reshape(xx.shape)
        ax.contourf(xx, yy, Z, levels=20, alpha=0.3)
        ax.contour(xx, yy, Z, levels=[0.5], colors="k", linewidths=1)
    else:
        Z = model.predict(grid).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.3)

    ax.scatter(X[:, 0], X[:, 1], c=y, s=60, edgecolor="k")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    return ax


## 3. Toy example: `make_moons` (2D)
This section mirrors many textbook plots: we can directly visualize decision boundaries in 2D.

In [ ]:
X, y = make_moons(n_samples=200, noise=0.25, random_state=3)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=42
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, s=60, edgecolor="k")
ax.set_title("Training data (make_moons)")
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
plt.show()


### 3.1 Baseline: logistic regression vs. MLP
Logistic regression is linear; an MLP can learn nonlinear boundaries. We use a pipeline so scaling is always applied consistently.

In [ ]:
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=5000, random_state=0)
)
mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(20,), solver="lbfgs", random_state=0, max_iter=2000)
)

log_reg.fit(X_train, y_train)
mlp.fit(X_train, y_train)

print(f"LogReg  train/test accuracy: {log_reg.score(X_train, y_train):.3f} / {log_reg.score(X_test, y_test):.3f}")
print(f"MLP(20) train/test accuracy: {mlp.score(X_train, y_train):.3f} / {mlp.score(X_test, y_test):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
plot_decision_boundary(log_reg, X_train, y_train, ax=axes[0], title="Logistic regression (scaled)")
plot_decision_boundary(mlp, X_train, y_train, ax=axes[1], title="MLP: 1 hidden layer (20 units, scaled)")
plt.tight_layout()
plt.show()


## 4. Architecture × regularization: a clean subplot grid
We vary:
- **Hidden units** per layer (rows)
- **L2 penalty** `alpha` (columns)

This uses `itertools.product` to generate all parameter combinations.

In [ ]:
import itertools

hidden_list = [10, 100]
alpha_list  = [0.0001, 0.01, 0.1, 1.0]

fig, axes = plt.subplots(
    len(hidden_list), len(alpha_list),
    figsize=(20, 8),
    sharex=True, sharey=True
)

for (i, n_hidden), (j, alpha) in itertools.product(enumerate(hidden_list), enumerate(alpha_list)):
    ax = axes[i, j]

    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            solver="lbfgs",
            random_state=0,
            hidden_layer_sizes=(n_hidden, n_hidden),
            alpha=alpha,
            max_iter=2000
        )
    )
    model.fit(X_train, y_train)

    # Plot boundary on *training* data for comparability across subplots
    plot_decision_boundary(model, X_train, y_train, ax=ax, title=None)

    # Clean, non-repetitive labeling
    if i == 0:
        ax.set_title(f"alpha={alpha:g}")
    if j == 0:
        ax.set_ylabel(f"hidden={n_hidden}\nfeature 2")
    if i == len(hidden_list) - 1:
        ax.set_xlabel("feature 1")

plt.suptitle("MLP decision boundaries: architecture (rows) × regularization alpha (cols)", y=1.02)
plt.tight_layout()
plt.show()


## 5. Activation functions (qualitative comparison)
We keep the architecture fixed and compare `relu`, `tanh`, and `logistic`.

In [ ]:
activations = ["relu", "tanh", "logistic"]
fig, axes = plt.subplots(1, len(activations), figsize=(18, 5), sharex=True, sharey=True)

for ax, act in zip(axes, activations):
    model = make_pipeline(
        StandardScaler(),
        MLPClassifier(
            solver="lbfgs",
            random_state=0,
            hidden_layer_sizes=(20, 20),
            activation=act,
            alpha=0.01,
            max_iter=2000
        )
    )
    model.fit(X_train, y_train)
    plot_decision_boundary(model, X_train, y_train, ax=ax, title=f"activation={act}")

plt.tight_layout()
plt.show()


## 6. High-dimensional example: Breast Cancer dataset
In >2D we cannot plot decision boundaries directly, but we can:
- standardize features
- evaluate train/test accuracy
- inspect the first-layer weight matrix as a diagnostic (in scaled feature space)


In [ ]:
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=0
)

model = make_pipeline(
    StandardScaler(),
    MLPClassifier(random_state=0, max_iter=3000)
)
model.fit(X_train, y_train)

print(f"MLP (scaled) train accuracy: {model.score(X_train, y_train):.3f}")
print(f"MLP (scaled) test  accuracy: {model.score(X_test, y_test):.3f}")

# Access the fitted MLP inside the pipeline for inspection
mlp = model.named_steps["mlpclassifier"]

plt.figure(figsize=(20, 5))
plt.imshow(mlp.coefs_[0], interpolation="none", aspect="auto", cmap="viridis")
plt.yticks(range(len(cancer.feature_names)), cancer.feature_names)
plt.xlabel("Hidden unit index")
plt.colorbar(label="Weight value (input → first hidden layer)")
plt.title("First-layer weights (trained on standardized features)")
plt.tight_layout()
plt.show()


## 7. Takeaways
- **Scaling matters** for MLPs; use a pipeline to avoid train/test mismatch.
- Larger networks can fit more complex boundaries, but **regularization (`alpha`)** controls overfitting.
- Activation choices change inductive bias; `relu` is often a strong default.
- Inspecting first-layer weights is a quick sanity check (interpretation is in *scaled* units).
